In [3]:
from google.colab import files
upload = files.upload()

Saving spam1.csv to spam1.csv


In [4]:
# Importing Libraries

import pandas as pd
import re
import nltk

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Download stopwords
nltk.download('stopwords')

# Load dataset
df = pd.read_csv("spam1.csv", encoding='latin-1')


# Keep required columns
df = df[['Label', 'Messages']]

# Clean text FIRST
# Clean labels
df['Label'] = (
    df['Label']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Convert labels
df['Label'] = df['Label'].replace({
    'ham': 0,
    'spam': 1
})

# Stopwords
stop_words = set(stopwords.words('english'))

# Cleaning function
def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

# Apply cleaning
df['Messages'] = df['Messages'].apply(clean_text)

# Vectorization
vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(df['Messages'])

y = df['Label']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Model
model = LogisticRegression(class_weight='balanced')

model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

# Report
print(classification_report(y_test, y_pred))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/tmp/ipykernel_651/3077396815.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Label'] = df['Label'].replace({


Accuracy: 0.9766816143497757
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       965
           1       0.94      0.88      0.91       150

    accuracy                           0.98      1115
   macro avg       0.96      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
# Custom prediction
sample = ["Sagar You won a free I-phone"]

sample_clean = [clean_text(text) for text in sample]

sample_vector = vectorizer.transform(sample_clean)

prediction = model.predict(sample_vector)

if prediction[0] == 1:
    print("Spam Mail")
else:
    print("Not Spam")